# P1 -- Phase 2 pilot: what does the action do INSIDE the predictor?

Run top to bottom on an **L4 GPU**. Two parts:
- **Phase 2a** (one bundled Franka scene): pipeline validation, re-read with the corrected
  estimators.
- **Phase 2b** (12 real DROID scenes from a pre-verified 1.4 MB sample in this repo): the
  statistical pilot with error bars, calibrated controls, and the power calculation.

Estimators follow the Phase-2a adversarial review (PRE_REGISTRATION.md section 10.1):
per-token centering, scale-invariant variance fractions (raw traces are confounded by
pre-LN norm growth), the **mode-s unfolding** as the only valid action-code-expansion test
(pooled rank has a linear null of q(K+1), not q), antithetic pairs for nonlinearity, and
per-block write attribution for localization. No TFDS, no Drive; the checkpoint downloads
to `/content` once per session.

## 0. Environment, repo, checkpoint

In [ ]:
import sys, os, types, torch
import torch.nn.functional as F
import numpy as np
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', dev)
try:
    import einops  # noqa
except Exception:
    get_ipython().system('pip -q install einops')
if not os.path.exists('vjepa2'):
    get_ipython().system('git clone --depth 1 https://github.com/facebookresearch/vjepa2.git')
import pathlib
pb = pathlib.Path('vjepa2/src/hub/backbones.py'); tb = pb.read_text()
pb.write_text(tb.replace('\"http://localhost:8300\"', '\"https://dl.fbaipublicfiles.com/vjepa2\"'))
URL = 'https://dl.fbaipublicfiles.com/vjepa2/vjepa2-ac-vitg.pt'; SIZE = 11760743310
CKPT = '/content/vjepa2-ac-vitg.pt'
if not (os.path.exists(CKPT) and os.path.getsize(CKPT) == SIZE):
    print('downloading 11.76 GB to /content (once per session, ~4 min)...')
    get_ipython().system('wget -q -c "$URL" -O "$CKPT"')
assert os.path.exists(CKPT) and os.path.getsize(CKPT) == SIZE, 'download incomplete; re-run this cell'
print('checkpoint OK:', round(os.path.getsize(CKPT)/1e9, 2), 'GB')


## 1. Write the tested `p1_lib.py` (66 analytic assertions pass, incl. the v4 estimators)

In [ ]:
P1_LIB_SRC = r'''"""p1_lib.py -- infrastructure for P1 (mechanistic analysis of the V-JEPA 2-AC predictor).

Sections:
  [SPEC]  the Section 6 library, reproduced verbatim (site discovery, hooks, geometry,
          subspace ablation, CEM planning, occlusion stimulus).
  [P1ADD] Phase-1 additions used by Arms A/B and by the Cui-theory (transition-operator)
          measurement. Each addition has an analytic unit test in test_p1_lib.py.

Style: no em dashes anywhere (per protocol).
"""
import contextlib, re
import numpy as np, torch, torch.nn as nn

# =====================================================================================
# [SPEC] Section 6, reproduced verbatim
# =====================================================================================

# ---------- site discovery (AMENDED 2026-07-23 after source verification) ----------
# The Section-6 single-arg discover_sites(model) returns ZERO sites on the real AC model,
# for two verified path-parsing reasons (NOT class naming, which was fine):
#   (a) the block regex required a literal ".blocks." but the real path is
#       "predictor_blocks.0" (no dot before "blocks");
#   (b) the family heuristic scanned the path for "encoder"/"trunk", but the standalone
#       encoder tree's own paths are just "blocks.0.attn".
# Also: torch.hub.load("facebookresearch/vjepa2","vjepa2_ac_vit_giant") returns a TUPLE
# (encoder, predictor) -- two separate trees -- so family MUST be passed explicitly.
_BLOCK_RE = re.compile(r"(?:^|\.)(?:predictor_blocks|blocks|layers|layer)\.(\d+)(?:\.|$)")
_COND_MODULES = ("action_encoder", "state_encoder", "extrinsics_encoder",
                 "predictor_embed", "predictor_proj", "predictor_norm")

def discover_sites(module, family):
    """Verified for facebookresearch/vjepa2 AC (VisionTransformerPredictorAC) at HEAD.
    `family` is EXPLICIT ('enc'|'prd') because the loader hands back two separate trees and
    neither path string identifies itself. Returns {name: (path, family, blk, kind)} with
    canonical names {fam}.blk{j}.{kind} plus non-block conditioning modules {fam}.{name}
    (blk = -1, kind = 'cond'). Call as discover_sites(predictor,'prd') and
    discover_sites(encoder,'enc'), then merge."""
    assert family in ("enc", "prd")
    sites = {}
    for path, mod in module.named_modules():
        m = _BLOCK_RE.search(path)
        if m is None: continue
        blk, cls = int(m.group(1)), type(mod).__name__.lower()
        kind = ("attn_out" if ("attention" in cls or cls.endswith("attn")) else
                "mlp_out"  if ("mlp" in cls or "swiglu" in cls or "feedforward" in cls) else
                "resid_post" if ("block" in cls or "layer" in cls) else None)
        if kind is None: continue
        sites.setdefault(f"{family}.blk{blk}.{kind}", (path, family, blk, kind))
    for path, mod in module.named_modules():
        if path in _COND_MODULES:
            sites.setdefault(f"{family}.{path}", (path, family, -1, "cond"))
    if not sites:
        raise RuntimeError(f"no sites for family={family}; print(module) and update _BLOCK_RE")
    return sites

def n_blocks(sites, fam):
    b = {v[2] for v in sites.values() if v[1]==fam}
    return max(b)+1 if b else 0

def get_module(model, dotted):
    m = model
    for p in dotted.split("."):
        m = m[int(p)] if p.isdigit() else getattr(m, p)
    return m

def find_action_pathway(model):
    return [p for p,_ in model.named_modules()
            if any(k in p.lower() for k in ("action","act_embed","cond"))]

# ---------- hooks ----------
def _tensor(o):
    if isinstance(o, torch.Tensor): return o
    if isinstance(o,(tuple,list)) and o: return _tensor(o[0])
    if hasattr(o,"last_hidden_state"): return o.last_hidden_state
    raise TypeError(type(o))

def _splice(o, new):
    if isinstance(o, torch.Tensor): return new
    if isinstance(o, tuple): return (new,)+tuple(o[1:])
    if isinstance(o, list):  return [new]+list(o[1:])
    o.last_hidden_state = new; return o

class Cache(contextlib.AbstractContextManager):
    def __init__(self, model, sites, names, to_cpu=True):
        self.model, self.sites, self.names, self.to_cpu = model, sites, names, to_cpu
        self.store, self._h = {}, []
    def __enter__(self):
        for n in self.names:
            def hk(_m,_i,o,n=n):
                t = _tensor(o).detach()
                self.store[n] = t.cpu() if self.to_cpu else t
                return o
            self._h.append(get_module(self.model, self.sites[n][0]).register_forward_hook(hk))
        return self
    def __exit__(self,*e):
        [h.remove() for h in self._h]; self._h.clear()
    def __getitem__(self,k): return self.store[k]

class SubspaceAblator(contextlib.AbstractContextManager):
    """Project OUT span(U) at each site: t <- t - U U^T t.
    Cleaner than whole-block ablation. NEVER use zero ablation as primary;
    it is off-distribution and reliably overstates importance."""
    def __init__(self, model, sites, bases):   # bases: {site: (d,k) orthonormal}
        self.model, self.sites, self.bases, self._h = model, sites, bases, []
    def __enter__(self):
        for n,U in self.bases.items():
            def hk(_m,_i,o,U=U):
                t = _tensor(o); Ud = U.to(t.device, t.dtype)
                return _splice(o, t - torch.einsum("...d,dk,ek->...e", t, Ud, Ud))
            self._h.append(get_module(self.model, self.sites[n][0]).register_forward_hook(hk))
        return self
    def __exit__(self,*e):
        [h.remove() for h in self._h]; self._h.clear()

# ---------- geometry ----------
def spectrum(x, center=True):
    a = x.detach().float().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)
    a = a.reshape(-1, a.shape[-1]).astype(np.float64)
    if center: a = a - a.mean(0, keepdims=True)
    s = np.linalg.svd(a, compute_uv=False)
    return np.clip(s**2 / max(a.shape[0]-1,1), 0, None)

def participation_ratio(lam):
    """Y2 = sum p_i^2, p_i = lam_i/sum lam. Y2->1 is condensation onto one
    direction; Y2 ~ 1/d is isotropic. Report 1/Y2 as an effective count."""
    lam = np.asarray(lam,float); t = lam.sum()
    return 1.0 if t<=0 else float(((lam/t)**2).sum())

def effective_rank(lam):
    lam = np.asarray(lam,float); t = lam.sum()
    if t<=0: return 1.0
    p = lam/t; p = p[p>0]
    return float(np.exp(-(p*np.log(p)).sum()))

def subspace_from_diffs(diffs, k=8):
    """Top-k principal directions of action-induced differences. (d,k) orthonormal."""
    A = diffs.reshape(-1, diffs.shape[-1]).float()
    A = A - A.mean(0, keepdim=True)
    _,_,Vh = torch.linalg.svd(A, full_matrices=False)
    return Vh[:k].T.contiguous()

def random_subspace(d, k, seed=0):
    g = torch.Generator().manual_seed(seed)
    Q,_ = torch.linalg.qr(torch.randn(d,k,generator=g)); return Q

def principal_angles(U, V):
    return torch.arccos(torch.linalg.svdvals(U.T @ V).clamp(-1,1))

def bootstrap_ci(v, stat=np.mean, n=10000, alpha=0.05, seed=0):
    rng = np.random.default_rng(seed); v = np.asarray(v,float); v = v[~np.isnan(v)]
    b = np.array([stat(rng.choice(v,v.size,replace=True)) for _ in range(n)])
    return float(stat(v)), float(np.percentile(b,100*alpha/2)), float(np.percentile(b,100*(1-alpha/2)))

# ---------- CEM planning (causal readout for Arm B) ----------
@torch.no_grad()
def cem_plan(predictor, z0, z_goal, horizon=8, n=512, elites=64, iters=6,
             action_dim=7, seed=0):
    g = torch.Generator(device=z0.device).manual_seed(seed)
    mu = torch.zeros(horizon, action_dim, device=z0.device)
    sd = torch.ones_like(mu)*0.5; trace=[]
    for _ in range(iters):
        a = (mu + sd*torch.randn((n,horizon,action_dim),generator=g,device=z0.device)).clamp(-1,1)
        z = z0.expand(n,*z0.shape[1:]).clone()
        for t in range(horizon): z = predictor(z, a[:,t])
        cost = (z - z_goal.expand_as(z)).pow(2).flatten(1).sum(1)
        idx = torch.topk(-cost, elites).indices
        w = torch.softmax(-cost[idx], 0).view(elites,1,1)
        mu = (w*a[idx]).sum(0)
        sd = ((w*(a[idx]-mu)**2).sum(0)).sqrt().clamp_min(1e-3)
        trace.append(float(cost.min()))
    return mu, trace

# ---------- occlusion stimulus generator (Arm D) ----------
class OccludedBounce(torch.utils.data.Dataset):
    """Object bounces with known dynamics; occluder hides it for occ_len frames.
    The object KEEPS MOVING behind the occluder, so a model with persistence
    should predict where it reappears."""
    def __init__(self, n=512, T=16, size=224, occ_start=5, occ_len=4, seed=0):
        self.n,self.T,self.S,self.o0,self.ol,self.seed = n,T,size,occ_start,occ_len,seed
    def __len__(self): return self.n
    def __getitem__(self, i):
        rng = np.random.default_rng(self.seed*1000003+i); S=self.S
        yy,xx = np.mgrid[0:S,0:S]/S
        x,y = rng.uniform(.2,.8,2); vx,vy = rng.uniform(-.08,.08,2); r=.10
        frames, traj = [], []
        for t in range(self.T):
            img = np.full((3,S,S), .75, np.float32)
            m = ((np.sqrt((xx-x)**2+(yy-y)**2) < r)).astype(np.float32)
            img = img*(1-m) + np.array([.9,.2,.2],np.float32)[:,None,None]*m
            if self.o0 <= t < self.o0+self.ol:
                img[:, :, int(.35*S):int(.65*S)] = .15
            frames.append(img); traj.append((x,y))
            x,y = x+vx, y+vy
            if not r<x<1-r: vx=-vx; x=float(np.clip(x,r,1-r))
            if not r<y<1-r: vy=-vy; y=float(np.clip(y,r,1-r))
        return {"pixel_values": torch.tensor(np.stack(frames)),
                "traj": torch.tensor(traj, dtype=torch.float32),
                "occluded": list(range(self.o0, min(self.o0+self.ol,self.T))),
                "video_id": i}

# =====================================================================================
# [P1ADD] Phase-1 additions. Each has an analytic unit test.
#
# Design note (the pivot of Phase 0): H1a operationalizes ACTION-INDUCED RESIDUAL-STREAM
# MODULATION -- object (b). Cui et al. 2606.27014 Thm 3.1 claim low rank of the TRANSITION
# OPERATOR M-bar(a) -- object (a). These are different objects. To speak to the theory you
# must call transition_operator_spectrum (object a); to characterize routing you call the
# action_diffs / participation_ratio path (object b). Never report one as the other.
# =====================================================================================

# ---------- Arm A: designed action sweep + matched-norm null ----------
def action_basis(action_dim=7, magnitudes=(0.5, 1.0), sign_flips=True, include_zero=True):
    """Deterministic designed action set for Arm A: signed, scaled canonical basis
    directions. Returns (M, action_dim). Row 0 is the zero action (the reference) when
    include_zero. No randomness here: the randomized comparator is matched_norm_null."""
    rows = []
    if include_zero:
        rows.append(torch.zeros(action_dim))
    signs = (1.0, -1.0) if sign_flips else (1.0,)
    for i in range(action_dim):
        e = torch.zeros(action_dim); e[i] = 1.0
        for m in magnitudes:
            for s in signs:
                rows.append(s * m * e)
    return torch.stack(rows)

def matched_norm_null(actions, seed=0):
    """MANDATORY Arm A baseline. Random-direction vectors whose per-row L2 norm matches
    `actions`. Same shape. Reports are always real/null, never raw (Section 5, Section 10)."""
    g = torch.Generator().manual_seed(seed)
    r = torch.randn(actions.shape, generator=g)
    rn = r / r.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return rn * actions.norm(dim=-1, keepdim=True)

def _keep_tokens(n_tok, exclude_positions):
    ex = set() if exclude_positions is None else set(int(t) for t in exclude_positions)
    return [t for t in range(n_tok) if t not in ex]

def action_induced_variance(acts, exclude_positions=None):
    """acts: (n_actions, n_tokens, d) residual-stream activations at ONE block, at FIXED
    state, one row per swept action. Returns trace of the across-action covariance averaged
    over kept token positions (the Arm A numerator). `exclude_positions` MUST include the
    action-token position(s): variance there is trivial and inflates the effect (confound).
    Report action_induced_variance(real) / action_induced_variance(null), never raw."""
    a = acts.detach().float()
    keep = _keep_tokens(a.shape[1], exclude_positions)
    a = a[:, keep, :]                                   # (n_act, n_keep, d)
    return float(a.var(dim=0, unbiased=True).sum(dim=-1).mean())

def effect_ratio(real, null, eps=1e-12):
    """Arm A / Arm B headline quantity: real over matched null."""
    return float(real / (null + eps))

def action_diffs(acts, ref_index=0, exclude_positions=None):
    """Stack the action-induced DIFFERENCE vectors (each swept action minus the reference
    action, pooled over kept tokens) into (n_act*n_keep, d). Feed to spectrum ->
    participation_ratio / effective_rank for the H1a routing-rank measurement (object b)."""
    a = acts.detach().float()
    keep = _keep_tokens(a.shape[1], exclude_positions)
    a = a[:, keep, :]
    d = a - a[ref_index:ref_index+1]
    return d.reshape(-1, a.shape[-1])

def effective_count(lam):
    """1/Y2, the participation-ratio effective count. Used for the H1a gate
    (effective count as a fraction of d)."""
    return 1.0 / max(participation_ratio(lam), 1e-12)

# ---------- Cui-theory object (a): transition-operator rank ----------
def transition_operator_spectrum(z_ctx, z_pred, center=True):
    """Singular values (descending) of the empirical cross-covariance between context
    embeddings z_ctx (..., d) and predicted-target embeddings z_pred (..., d') at FIXED
    action. This is the object Cui et al. 2606.27014 call low-rank (the transition kernel
    M-bar(a)). Its effective rank -- NOT the residual-stream rank -- is what a refutation of
    the low-rank theory (outcome cell 5) must target."""
    A = (z_ctx.detach().float().cpu().numpy() if torch.is_tensor(z_ctx) else np.asarray(z_ctx))
    B = (z_pred.detach().float().cpu().numpy() if torch.is_tensor(z_pred) else np.asarray(z_pred))
    A = A.reshape(-1, A.shape[-1]).astype(np.float64)
    B = B.reshape(-1, B.shape[-1]).astype(np.float64)
    if center:
        A = A - A.mean(0, keepdims=True); B = B - B.mean(0, keepdims=True)
    C = (A.T @ B) / max(A.shape[0]-1, 1)
    return np.linalg.svd(C, compute_uv=False)

# ---------- Arm B: localization ----------
def cumulative_localization(per_block_effect, frac=0.80):
    """per_block_effect: array of (real - null-corrected) action effect per predictor block,
    IN BLOCK ORDER. Returns (cum_blockorder, n_for_frac, contiguous_span) where
      cum_blockorder = cumulative fraction along block order (for the H1b plot),
      n_for_frac     = minimal number of blocks (highest-effect first) reaching `frac`
                       of total effect  --> the H1b gate quantity,
      contiguous_span= size of the smallest contiguous block window holding >= frac
                       (H1b's 'contiguous minority' reading)."""
    e = np.clip(np.asarray(per_block_effect, float), 0, None)
    tot = max(e.sum(), 1e-12)
    cum_blockorder = np.cumsum(e) / tot
    order = np.argsort(-e)
    n_for_frac = int(np.searchsorted(np.cumsum(e[order]) / tot, frac) + 1)
    # smallest contiguous window reaching frac
    nb = len(e); best = nb
    for i in range(nb):
        s = 0.0
        for j in range(i, nb):
            s += e[j]
            if s / tot >= frac:
                best = min(best, j - i + 1); break
    return cum_blockorder, n_for_frac, int(best)

# =====================================================================================
# [P1ADD-v2] Arm A hardening (mandated by the Phase-0 adversarial review).
#
# The Section-5 null ("random vector of matched norm") is OFF the action-embedding
# manifold: passed as a raw residual perturbation it under-propagates through attention/MLP
# trained to read the thin action subspace, which DEFLATES the null and INFLATES real/null.
# The correct nulls are on-manifold: random RAW actions embedded through the SAME action
# input head, or PERMUTED real actions. Both are provided. Also: PR alone is a low-biased
# rank estimator, so report a panel of estimators; and finite differences can cancel
# multiplicative action*state directions, so a Jacobian-SVD read is provided as a check.
# =====================================================================================

def permuted_action_null(actions, seed=0):
    """Strongest Arm A null: a permutation of the REAL swept actions. Preserves the exact
    action-embedding statistics (every row is a genuine action) while breaking the
    action<->state correspondence. Feed these through the real action head, same as `actions`.
    Returns a permuted copy (guaranteed != identity for n>1)."""
    n = actions.shape[0]
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n, generator=g)
    if n > 1:                                        # avoid the identity permutation
        tries = 0
        while bool((perm == torch.arange(n)).all()) and tries < 16:
            perm = torch.randperm(n, generator=g); tries += 1
    return actions[perm]

def random_action_null(action_dim, ref_norms, seed=0):
    """On-manifold null: random RAW action vectors (in R^{action_dim}) whose per-row norm
    matches `ref_norms`. MUST be embedded through the model's real action input head so it
    traverses the trained pathway (this is what makes it drive-matched, unlike a raw
    R^width residual perturbation). ref_norms: (n,) tensor of target action-space norms."""
    ref_norms = ref_norms.reshape(-1)
    g = torch.Generator().manual_seed(seed)
    r = torch.randn(ref_norms.shape[0], action_dim, generator=g)
    rn = r / r.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return rn * ref_norms[:, None]

def dense_action_sample(action_dim, n, low=-1.0, high=1.0, seed=0):
    """Dense on-distribution action sampling for the PR-vs-N plateau check (fixes the
    designed-basis rank ceiling: k designed actions span <= k dims, so PR<=k trivially).
    Report PR vs N; if PR keeps rising with N you are sample-limited, not measuring the
    pathway. Uniform over the valid box [low,high]^action_dim by default."""
    g = torch.Generator().manual_seed(seed)
    return low + (high - low) * torch.rand(n, action_dim, generator=g)

# ---------- rank-estimator panel (report all; never PR alone) ----------
def stable_rank(lam):
    """sum(sigma_i^2)/sigma_max^2 = sum(lam)/max(lam) for eigenvalues lam=sigma^2.
    Less top-eigenvalue-dominated than PR."""
    lam = np.asarray(lam, float); lam = lam[lam > 0]
    return float(lam.sum() / lam.max()) if lam.size and lam.max() > 0 else 0.0

def numerical_rank(lam, rel_thresh=1e-2):
    """Count of eigenvalues above rel_thresh * max eigenvalue."""
    lam = np.asarray(lam, float)
    return int((lam > rel_thresh * (lam.max() if lam.size else 0)).sum())

def count_above_floor(lam, floor):
    """Count of eigenvalues above an absolute floor (e.g. the on-manifold null's spectrum).
    This is the on-manifold-null-referenced rank the low-rank claim should use."""
    return int((np.asarray(lam, float) > float(floor)).sum())

def rank_report(lam, null_floor=None, rel_thresh=1e-2):
    """Full panel for one spectrum. Report ALL of these, not PR alone (PR is low-biased:
    one dominant eigenvalue drives it to ~1)."""
    lam = np.asarray(lam, float)
    r = {
        "participation_effective_count": effective_count(lam),   # 1/Y2
        "entropy_effective_rank": effective_rank(lam),
        "stable_rank": stable_rank(lam),
        "numerical_rank": numerical_rank(lam, rel_thresh),
        "top_eigenvalue_frac": float(lam.max() / lam.sum()) if lam.sum() > 0 else 0.0,
    }
    if null_floor is not None:
        r["rank_above_null_floor"] = count_above_floor(lam, null_floor)
    return r

# ---------- analytic Jacobian read (local action-routing rank) ----------
def jacobian_spectrum(fn, a0, center=None):
    """Singular values of d fn / d a evaluated at a0, via autodiff. fn maps an action-space
    vector (shape (action_dim,)) to a flattened residual read (shape (m,)). Complements the
    finite-difference read: a two-forward-pass difference locally linearizes GELU and can
    cancel multiplicative action*state directions, understating rank; the Jacobian does not.
    Returns singular values (descending) of the (m x action_dim) Jacobian."""
    a0 = a0.detach().clone().requires_grad_(True)
    J = torch.autograd.functional.jacobian(fn, a0)           # (m, action_dim)
    J = J.reshape(-1, a0.numel()).double()
    return torch.linalg.svdvals(J).cpu().numpy()

# =====================================================================================
# [P1ADD-v3] AC token layout + gate assertions (from source verification 2026-07-23).
#
# The action is NOT an additive residual modulation. VisionTransformerPredictorAC interleaves
# it as a dedicated token:  x = cat([action, state, patches], dim=2).flatten(1,2). Per frame:
#   [action, state, patch_0 .. patch_{HW-1}], stride HW+K (K=2, or 3 with extrinsics).
# So Arm A reads the action effect on PATCH tokens only, excluding the per-frame cond tokens,
# and the routing object is the action-token -> patch-token ATTENTION-mediated update.
# There are TWO 7-d conditioning streams (action AND state); a state-swap control (vary state,
# hold action) is a mandatory second null, else action-attributable variance is confounded
# with proprio. Attention is frame-causal over the FULL history (frame t1 attends to all
# t2 <= t1); perturbing the action at frame t must leave every token in frames < t exactly
# unchanged -- a hard Phase-1 gate assertion.
# =====================================================================================

def token_index(T, H=16, W=16, K=2):
    """AC interleave map. Returns a dict of LongTensors: 'action', 'state', 'cond'
    (all conditioning tokens, sorted), 'patch'. For 256px/patch16: H=W=16, HW=256, K=2,
    stride=258 tokens per frame."""
    HW = H * W; stride = HW + K
    action = torch.tensor([t * stride + 0 for t in range(T)])
    state  = torch.tensor([t * stride + 1 for t in range(T)])
    cond   = torch.tensor([t * stride + c for t in range(T) for c in range(K)])
    patch  = torch.tensor([t * stride + K + i for t in range(T) for i in range(HW)])
    return {"action": action, "state": state, "cond": cond.sort().values, "patch": patch}

def cond_positions(T, H=16, W=16, K=2):
    """Per-frame conditioning tokens to EXCLUDE from Arm A readouts (their variance is
    trivially driven by the swept input)."""
    return token_index(T, H, W, K)["cond"]

def patch_positions(T, H=16, W=16, K=2):
    """The tokens Arm A reads (patch tokens; the action reaches them only through attention)."""
    return token_index(T, H, W, K)["patch"]

def empirical_action_null(action_pool, n, seed=0):
    """PRIMARY on-manifold null (site-verification finding 4). Resample n action VECTORS with
    replacement from the empirical action distribution `action_pool` (M, action_dim) of REAL
    actions, to be embedded through the real action_encoder. Because the action enters through
    a learned 7->1024 encoder, the encoder-OUTPUT manifold is what matters, and real (or
    resampled-real) actions are the only guaranteed-on-manifold null. Preferred over
    random_action_null / matched_norm_null."""
    g = torch.Generator().manual_seed(seed)
    idx = torch.randint(action_pool.shape[0], (n,), generator=g)
    return action_pool[idx]

@torch.no_grad()
def assert_frame_causal(predictor, B=2, T=4, HW=256, d_enc=1408,
                        action_dim=7, state_dim=7, perturb_frame=2):
    """Phase-1 structural gate. Perturbing the action at frame `perturb_frame` must produce
    EXACTLY zero change at every token in earlier frames, and a nonzero change at that frame.
    Catches hook misplacement, wrong token indexing, and mask misuse in one forward pass.
    Assumes predictor(x, actions, states) with x:(B,T*HW,d_enc), actions/states:(B,T,dim)."""
    x = torch.randn(B, T * HW, d_enc)
    a = torch.randn(B, T, action_dim); s = torch.randn(B, T, state_dim)
    out0 = predictor(x, a, s)
    a2 = a.clone(); a2[:, perturb_frame] += 5.0
    out1 = predictor(x, a2, s)
    d = (out1 - out0).abs().reshape(B, T, HW, -1).amax(dim=(0, 2, 3))
    for t in range(perturb_frame):
        assert float(d[t]) == 0, f"CAUSALITY VIOLATED at frame {t}: {float(d[t]):.3e}"
    assert float(d[perturb_frame]) > 0, "action had no forward effect; wiring is wrong"
    return {"per_frame_max_delta": [float(v) for v in d], "passed": True}

# =====================================================================================
# [P1ADD-v4] Estimators mandated by the Phase-2a adversarial review (2026-07-23).
#
# Constructively shown by simulation during the review: pooled (action x token) diff rank
# exceeding action_dim carries NO evidence of action-code expansion; a strictly linear
# 7-dim-code model with token-diverse linear readouts (d_st = G_t J a_s) reproduces pooled
# PR 50-108, because the linear null for the pooled statistic is q(K+1) (K = token-context
# dims), not q. The statistic that tests code expansion is the mode-s unfolding below.
# Raw variance across depth is confounded by pre-LN residual norm growth (~1.3x/block);
# only fractions and LN-normalized variants are comparable across blocks. Grand-mean
# centering leaves the (rank <= T-1) spatial footprint inside the pooled covariance;
# center per token. H1b localization must be judged on per-block WRITES (h_l - h_{l-1}).
# =====================================================================================

def per_token_centered(acts, exclude_positions=None):
    """(n_act, n_tok, d) -> (n_act, n_keep, d) with EACH kept token's mean over actions
    removed. Removes the between-token mean footprint that grand-mean centering leaves in."""
    a = acts.detach().float()
    keep = _keep_tokens(a.shape[1], exclude_positions)
    a = a[:, keep, :]
    return a - a.mean(dim=0, keepdim=True)

def pooled_diff_spectrum(acts, exclude_positions=None):
    """PRIMARY pooled spectrum: per-token-centered, pooled over (action x token).
    CAUTION (linear null): rank here caps at q(K+1) under a q-dim code with token-diverse
    linear readouts; exceeding q is NOT evidence of code expansion. Use mode_s_spectrum
    for that question."""
    c = per_token_centered(acts, exclude_positions)
    return spectrum(c.reshape(-1, c.shape[-1]), center=False)

def mode_s_spectrum(acts, exclude_positions=None):
    """Mode-s unfolding: rows = actions, columns = concatenated (token, dim) coordinates,
    eigenvalues via the S x S Gram (cheap). Under ANY model d_st = R_t c(a_s) with a q-dim
    code and arbitrary per-token linear readouts, centered rank <= q. THIS is the code-
    expansion test. Capped at S-1: report alongside an S-saturation check."""
    c = per_token_centered(acts, exclude_positions)      # = row(action)-centering of M
    M = c.reshape(c.shape[0], -1).double()
    G = M @ M.T / max(M.shape[0] - 1, 1)
    lam = torch.linalg.eigvalsh(G).clamp_min(0).flip(0)
    return lam.cpu().numpy()

def even_part_fraction(diff_plus, diff_minus, exclude_positions=None):
    """Antithetic pairs. diff_plus/minus: (n, n_tok, d) diffs vs the SAME zero-action
    reference for actions +a and -a (matched rows). ||(d(a)+d(-a))/2||^2 over mean||d||^2.
    Exactly 0 for any response linear in a; a direct meter of even-order nonlinearity."""
    p = diff_plus.detach().float(); m = diff_minus.detach().float()
    keep = _keep_tokens(p.shape[1], exclude_positions)
    p = p[:, keep, :].reshape(p.shape[0], -1)
    m = m[:, keep, :].reshape(m.shape[0], -1)
    even = 0.5 * (p + m)
    denom = 0.5 * (p.pow(2).sum(-1) + m.pow(2).sum(-1))
    return float((even.pow(2).sum(-1) / denom.clamp_min(1e-12)).mean())

def variance_fraction(acts, exclude_positions=None):
    """The protocol's Arm A metric: across-action variance (mean over kept tokens of the
    per-token across-action variance trace) as a FRACTION of total pooled variance.
    Scale-invariant; the only depth-comparable form of the variance readout."""
    a = acts.detach().float()
    keep = _keep_tokens(a.shape[1], exclude_positions)
    a = a[:, keep, :]
    num = float(a.var(dim=0, unbiased=True).sum(dim=-1).mean())
    A = a.reshape(-1, a.shape[-1])
    tot = float(A.var(dim=0, unbiased=True).sum())
    return num / max(tot, 1e-12)

def ln_normalized(acts, eps=1e-6):
    """Parameter-free per-vector LayerNorm. The read-relevant view of the residual stream:
    every consumer (next block pre-LN, predictor_norm) reads through LayerNorm, so raw
    late-block magnitude is invisible to the output."""
    a = acts.detach().float()
    return (a - a.mean(-1, keepdim=True)) / (a.std(-1, keepdim=True) + eps)

@torch.no_grad()
def assert_real_checkpoint(encoder, predictor,
                           predictor_depth=24, action_dim=7, init_std=0.02):
    """Protocol R7 aid (NOT a substitute for the reserved human attestation H1). Run ONCE
    against genuinely loaded weights before any claim. The std heuristic flags an
    action_encoder still at trunc_normal_(std=init_std) init, i.e. a state_dict that did not
    populate the conditioning pathway."""
    assert type(predictor).__name__ == "VisionTransformerPredictorAC", type(predictor).__name__
    assert len(predictor.predictor_blocks) == predictor_depth
    assert predictor.action_encoder.in_features == action_dim
    assert predictor.predictor_embed.in_features == encoder.embed_dim
    assert getattr(predictor, "is_frame_causal", True) and not getattr(predictor, "use_extrinsics", False)
    w = predictor.action_encoder.weight.detach()
    assert w.std().item() > init_std + 0.005, (
        f"action_encoder.weight std={w.std().item():.4f} looks like fresh init "
        f"(~{init_std}); state_dict did not populate the conditioning pathway")
    return {"loaded": True, "action_encoder_std": float(w.std())}
'''
open('p1_lib.py','w').write(P1_LIB_SRC)
import importlib, p1_lib; importlib.reload(p1_lib)
print('p1_lib fns:', len([k for k in dir(p1_lib) if not k.startswith('_')]))


## 2. Build encoder + predictor, load weights (mmap)

In [ ]:
def _drop_path(x, drop_prob=0., training=False, scale_by_keep=True):
    if drop_prob == 0. or not training: return x
    keep = 1 - drop_prob; shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    r = x.new_empty(shape).bernoulli_(keep)
    if keep > 0.0 and scale_by_keep: r.div_(keep)
    return x * r
for _n in ('timm', 'timm.models', 'timm.models.layers'):
    sys.modules.setdefault(_n, types.ModuleType(_n))
sys.modules['timm.models.layers'].drop_path = _drop_path
sys.path.insert(0, 'vjepa2')
from src.models.ac_predictor import vit_ac_predictor
from src.models.vision_transformer import vit_giant_xformers
sd = torch.load(CKPT, map_location='cpu', mmap=True, weights_only=False)
def clean(d): return {k.replace('module.','').replace('backbone.',''): v for k,v in d.items()}
predictor = vit_ac_predictor(embed_dim=1408, img_size=(256,256), patch_size=16, num_frames=64, tubelet_size=2)
pinfo = predictor.load_state_dict(clean(sd['predictor']), strict=False)
encoder = vit_giant_xformers(patch_size=16, img_size=(256,256), num_frames=64, tubelet_size=2,
                             use_sdpa=True, use_SiLU=False, wide_SiLU=True, uniform_power=False, use_rope=True)
einfo = encoder.load_state_dict(clean(sd['encoder']), strict=False)
predictor = predictor.to(dev).eval(); encoder = encoder.to(dev).eval()
assert not pinfo.missing_keys and not pinfo.unexpected_keys
prd_sites = p1_lib.discover_sites(predictor, 'prd')
# trivial-gain baseline (review mandate): learned input-head Frobenius norms
Wa = predictor.action_encoder.weight.detach(); Ws = predictor.state_encoder.weight.detach()
print('loaded. sites', len(prd_sites), '| ||W_action||_F', round(float(Wa.norm()),3),
      '| ||W_state||_F', round(float(Ws.norm()),3),
      '| Frobenius energy ratio', round(float(Wa.norm()**2/Ws.norm()**2),3))


## 3. Shared machinery (v4 estimators)

All spectra: per-token centered, on LN-normalized activations (the read-relevant view).
All variance readouts: scale-invariant fractions. Localization: per-block writes.

In [ ]:
from app.vjepa_droid.transforms import make_transforms
from notebooks.utils.mpc_utils import poses_to_diff
crop = 256; tokens_per_frame = (crop // encoder.patch_size) ** 2
transform = make_transforms(random_horizontal_flip=False, random_resize_aspect_ratio=(1.,1.),
                            random_resize_scale=(1.,1.), reprob=0., auto_augment=False,
                            motion_shift=False, crop_size=crop)
T = 1; K = 2
cond = p1_lib.cond_positions(T, 16, 16, K)
names = [f'prd.blk{j}.resid_post' for j in range(24)]

@torch.no_grad()
def forward_target(c, normalize_reps=True):
    B,C,Tt,H,W = c.size()
    c = c.permute(0,2,1,3,4).flatten(0,1).unsqueeze(2).repeat(1,1,2,1,1)
    h = encoder(c); h = h.view(B,Tt,-1,h.size(-1)).flatten(1,2)
    return F.layer_norm(h,(h.size(-1),)) if normalize_reps else h

@torch.no_grad()
def encode_frame(frame_uint8):
    clip = transform(np.repeat(frame_uint8[None], 2, axis=0)).unsqueeze(0).to(dev)
    return forward_target(clip)[:, :tokens_per_frame]

@torch.no_grad()
def sweep_cache(z0, A, s):
    # z0 (1,tpf,1408); A (S,7); s (1,1,7) fixed or (S,7)/(S,1,7) per-row
    S = A.shape[0]
    zB = z0.repeat(S,1,1); aB = A[:,None].to(dev)
    sB = s.to(dev)
    if sB.dim() == 2: sB = sB[:, None]
    if sB.shape[0] == 1: sB = sB.repeat(S,1,1)
    with p1_lib.Cache(predictor, prd_sites, names, to_cpu=True) as C:
        _ = predictor(zB, aB, sB)
        return {n: C[n].float() for n in names}

def per_block_metrics(cache):
    out, prev = [], None
    for j in range(24):
        a = cache[f'prd.blk{j}.resid_post']
        aln = p1_lib.ln_normalized(a)
        row = dict(block=j,
                   vf_raw=p1_lib.variance_fraction(a, exclude_positions=cond),
                   vf_ln=p1_lib.variance_fraction(aln, exclude_positions=cond),
                   raw_var=p1_lib.action_induced_variance(a, exclude_positions=cond))
        lam_p = p1_lib.pooled_diff_spectrum(aln, exclude_positions=cond)
        row['pooled_eff'] = p1_lib.effective_count(lam_p)
        lam_s = p1_lib.mode_s_spectrum(aln, exclude_positions=cond)
        row['mode_s_eff'] = p1_lib.effective_count(lam_s)
        row['mode_s_rank'] = p1_lib.numerical_rank(lam_s, rel_thresh=1e-2)
        row['write_vf'] = (p1_lib.variance_fraction(a - prev, exclude_positions=cond)
                           if prev is not None else float('nan'))
        prev = a
        out.append(row)
    return out

def even_by_block(cache_plus, cache_minus):
    # both sweeps have row 0 = the reference (zero action); rows i>0 are +a_i / -a_i
    ev = []
    for j in range(24):
        p = cache_plus[f'prd.blk{j}.resid_post']; m = cache_minus[f'prd.blk{j}.resid_post']
        ev.append(p1_lib.even_part_fraction(p[1:]-p[0:1], m[1:]-m[0:1], exclude_positions=cond))
    return ev
print('machinery ready')


# Phase 2a -- one bundled Franka scene (pipeline validation, corrected estimators)

In [ ]:
traj = np.load('vjepa2/notebooks/franka_example_traj.npz')
np_clips, np_states = traj['observations'], traj['states']
z0 = encode_frame(np_clips[0,0])
s0 = torch.tensor(np_states[:, :1]).float()
a_real = poses_to_diff(np_states[0,0], np_states[0,1]).float()
rng = 0.075; S = 32
A  = torch.cat([torch.zeros(1,7), p1_lib.dense_action_sample(7, S, -rng, rng)])
cp = sweep_cache(z0, A, s0); cm = sweep_cache(z0, torch.cat([A[0:1], -A[1:]]), s0)
ma = per_block_metrics(cp); ev = even_by_block(cp, cm)
S_swap = torch.cat([s0.squeeze(1), s0.squeeze(1) + p1_lib.dense_action_sample(7, S, -rng, rng)])
ms = per_block_metrics(sweep_cache(z0, a_real[None].repeat(S+1,1), S_swap))
del cm
print('blk   vf_ln    act/state  mode_s_rank  pooled_eff  even%')
for j in range(24):
    r = ma[j]; ratio = r['vf_ln']/max(ms[j]['vf_ln'],1e-9)
    print(f"{j:3d}  {r['vf_ln']:.4f}  {ratio:8.2f}   {r['mode_s_rank']:5d}      {r['pooled_eff']:7.2f}   {100*ev[j]:.2f}")


## What Phase 2a shows (corrected)

Read with the corrected estimators: `vf_ln` is the depth-comparable action-variance
fraction (raw traces are norm-growth-confounded and no longer reported across depth);
`mode_s_rank` is the only valid code-expansion readout (> 7 needed to claim expansion;
pooled numbers have a q(K+1) linear null); `even%` meters genuine nonlinearity (0 for any
linear response). The act/state ratio here is still the UNCALIBRATED box comparison and is
read as a control only, not as routing evidence; the calibrated version is in 2b.

# Phase 2b -- 12 real DROID scenes (statistical pilot)

Data: `p1_droid_pilot_v1.npz` from this repo -- extracted locally from the public DROID
sample with schema verification (12 episodes, first exterior frame + full 7-dim proprio
series). Actions are rotation-correct pose diffs (`poses_to_diff`), so the sweep is the
EMPIRICAL action distribution. Controls: resampled REAL states (on-manifold), and
matched-norm random actions through the real head (conservative comparator).

## 2b.0 Load the verified sample and build empirical pools

In [ ]:
get_ipython().system('wget -q -c https://raw.githubusercontent.com/grewalsk/p1-vjepa2-predictor/main/p1_droid_pilot_v1.npz')
z = np.load('p1_droid_pilot_v1.npz')
frames, states, ep_ids = z['frames'], z['states'], z['ep_ids']
n_ep = int(z['n_episodes'])
print('frames', frames.shape, '| states', states.shape, '| episodes', n_ep)
# rotation-correct empirical action pool (consecutive within-episode pose diffs)
pool = []
for e in range(n_ep):
    s_e = states[ep_ids == e]
    for t in range(len(s_e) - 1):
        pool.append(poses_to_diff(s_e[t], s_e[t+1]).numpy())
action_pool = torch.tensor(np.stack(pool), dtype=torch.float32)
state_pool  = torch.tensor(states, dtype=torch.float32)
print('action pool', tuple(action_pool.shape), '| per-dim std:', [round(float(v),5) for v in action_pool.std(0)])
print('state  pool', tuple(state_pool.shape),  '| per-dim std:', [round(float(v),5) for v in state_pool.std(0)])


## 2b.1 Per-scene sweeps (5 per scene)

Per scene: (1) empirical actions vs zero ref, (2) their antithetic negatives, (3) empirical
actions around the scene's real action (operating-point robustness), (4) resampled REAL
states at fixed real action (on-manifold state null), (5) matched-norm random actions
through the real head (conservative null). Embedding step energies are recorded for the
norm-matched excess ratio.

In [ ]:
S = 32; N_SCENES = n_ep
scene_rows = []
for i in range(N_SCENES):
    s_e = states[ep_ids == i]
    z0 = encode_frame(frames[i]); s0 = torch.tensor(s_e[0])[None,None].float()
    a_real = poses_to_diff(s_e[0], s_e[min(5, len(s_e)-1)]).float()
    Ai  = p1_lib.empirical_action_null(action_pool, S, seed=100+i)
    A0  = torch.cat([torch.zeros(1,7), Ai])                       # ref = zero action
    A0m = torch.cat([torch.zeros(1,7), -Ai])                      # antithetic
    A1  = torch.cat([a_real[None], a_real[None] + Ai])            # ref = a_real (op point 2)
    Sr  = torch.cat([s0.squeeze(1), p1_lib.empirical_action_null(state_pool, S, seed=200+i)])
    An  = torch.cat([torch.zeros(1,7), p1_lib.random_action_null(7, Ai.norm(dim=-1), seed=300+i)])
    # EMBEDDING-NORM-MATCHED state sweep (review mandate 17, matched by construction):
    # directions = real-state-diff directions (locally on-manifold), rescaled per pair so
    # ||W_s ds_i|| == ||W_a a_i||. This is the parallel-gate comparator; no division needed.
    with torch.no_grad():
        Dl = (Sr[1:] - s0.squeeze(1))                                  # real-diff directions
        emb_s = (Dl @ Ws.cpu().T).norm(dim=-1).clamp_min(1e-12)
        emb_a = (Ai @ Wa.cpu().T).norm(dim=-1)
        Snm = torch.cat([s0.squeeze(1), s0.squeeze(1) + Dl * (emb_a / emb_s)[:, None]])
        row_emb_r = float(emb_a.pow(2).mean() / (Dl @ Ws.cpu().T).pow(2).sum(-1).mean())
    cp = sweep_cache(z0, A0, s0)
    cm = sweep_cache(z0, A0m, s0)
    row = dict(act=per_block_metrics(cp), even=even_by_block(cp, cm))
    del cp, cm
    row['act_op2']  = per_block_metrics(sweep_cache(z0, A1, s0))
    row['state']    = per_block_metrics(sweep_cache(z0, a_real[None].repeat(S+1,1), Sr))
    row['state_nm'] = per_block_metrics(sweep_cache(z0, a_real[None].repeat(S+1,1), Snm))
    row['mnull']    = per_block_metrics(sweep_cache(z0, An, s0))
    row['emb_energy_ratio'] = row_emb_r
    scene_rows.append(row)
    print(f'scene {i+1}/{N_SCENES} done (raw emb energy ratio {row_emb_r:.3e}; state_nm sweep is matched to 1 by construction)')
print('all scenes swept')


## 2b.2 Pool across scenes: gates, calibrated ratios, power

In [ ]:
def col(condn, key):
    return np.array([[s[condn][j][key] for j in range(24)] for s in scene_rows])
vf_act, vf_state, vf_mn = col('act','vf_ln'), col('state','vf_ln'), col('mnull','vf_ln')
vf_op2, vf_snm = col('act_op2','vf_ln'), col('state_nm','vf_ln')
raw_act, raw_state = col('act','raw_var'), col('state','raw_var')
modes_rank = col('act','mode_s_rank'); modes_eff = col('act','mode_s_eff')
pooled_eff = col('act','pooled_eff'); write_vf = col('act','write_vf')
even = np.array([s['even'] for s in scene_rows])
emb_r = np.array([s['emb_energy_ratio'] for s in scene_rows])

peak = int(np.nanmean(vf_act, 0).argmax())
print(f'peak vf_ln block: {peak}')
ratio_mn    = vf_act / np.clip(vf_mn, 1e-9, None)               # original gate comparator
ratio_state = vf_act / np.clip(vf_state, 1e-9, None)            # uncalibrated control (huge state jumps)
excess      = vf_act / np.clip(vf_snm, 1e-9, None)              # parallel gate: embedding-norm-MATCHED by construction
for tag, arr in [('vf_ln(action) @peak', vf_act[:,peak]),
                 ('act/matched-norm-null @peak (GATE >= 3.0)', ratio_mn[:,peak]),
                 ('act/state uncalibrated @peak (control only)', ratio_state[:,peak]),
                 ('act/state EMB-NORM-MATCHED @peak (parallel GATE >= 3.0)', excess[:,peak]),
                 ('mode-s rank @peak (code dims; 7 = no expansion)', modes_rank[:,peak]),
                 ('even-part fraction @peak (0 = linear)', even[:,peak])]:
    m, lo, hi = p1_lib.bootstrap_ci(arr, n=5000)
    print(f'  {tag:48s} mean {m:8.3f}  95% CI [{lo:.3f}, {hi:.3f}]')
# operating-point robustness (mandated): same quantities at op point a_real
m2, lo2, hi2 = p1_lib.bootstrap_ci((vf_op2/np.clip(vf_mn,1e-9,None))[:,peak], n=5000)
print(f'  act@a_real /mnull @peak (op-point 2)            mean {m2:8.3f}  95% CI [{lo2:.3f}, {hi2:.3f}]')
# power for the original gate
x = ratio_mn[:,peak]; mu, sdv = float(np.nanmean(x)), float(np.nanstd(x, ddof=1) + 1e-9)
eff = (mu - 3.0) / sdv
print(f'POWER vs gate 3.0: effect size d = {eff:.2f}; ~scenes for 80% power = '
      f'{(2.8/eff)**2:.0f}' if eff > 0 else f'POWER: mean ratio {mu:.2f} does not clear 3.0')


In [ ]:
import matplotlib.pyplot as plt
bl = list(range(24))
fig, ax = plt.subplots(1, 4, figsize=(20,4))
def band(a, arr, label):
    mean = np.nanmean(arr,0); lo = np.nanpercentile(arr,2.5,0); hi = np.nanpercentile(arr,97.5,0)
    a.plot(bl, mean, 'o-', ms=3, label=label); a.fill_between(bl, lo, hi, alpha=0.2)
band(ax[0], vf_act,'action'); band(ax[0], vf_snm,'state (emb-norm-matched)'); band(ax[0], vf_mn,'matched-norm null')
ax[0].set_title('LN variance fraction (scale-invariant)'); ax[0].set_xlabel('block'); ax[0].legend(); ax[0].set_yscale('log')
band(ax[1], write_vf,'action write vf')
ax[1].set_title('per-block WRITE attribution (H1b lives here)'); ax[1].set_xlabel('block')
band(ax[2], modes_rank,'mode-s rank'); band(ax[2], pooled_eff,'pooled eff (q(K+1) null)')
ax[2].axhline(7, ls='--', c='red', label='7 = no code expansion'); ax[2].set_title('code rank vs pooled rank'); ax[2].set_xlabel('block'); ax[2].legend()
band(ax[3], 100*even,'even part %')
ax[3].set_title('nonlinearity meter (0 = linear in action)'); ax[3].set_xlabel('block')
plt.tight_layout(); plt.savefig('phase2b_pilot.png', dpi=110); plt.show()
print('saved phase2b_pilot.png')


## What to paste back

From 2a: the printed per-block table. From 2b: the `peak`/GATE/POWER prints and
`phase2b_pilot.png`. Read as a **pilot** (the protocol forbids reporting the pilot point
estimate as evidence): we want (1) controls behaving -- action clears the matched-norm null,
and the norm-matched EXCESS (not the uncalibrated ratio) says whether anything beyond input
gain is going on; (2) the mode-s rank verdict -- 7 means no code expansion (the pooled
numbers exceeding 7 are expected under the linear null and prove nothing); (3) the even-part
fraction -- how nonlinear the action response actually is; (4) the write-attribution profile,
which is where H1b localization will be judged; (5) the power number for the main run.